In [0]:
select * from bikestore_project.bronze.orders

order_id,customer_id,order_status,order_date,required_date,shipped_date,store_id,staff_id
1,259,4,2016-01-01,2016-01-03,2016-01-03,1,2
2,1212,4,2016-01-01,2016-01-04,2016-01-03,2,6
3,523,4,2016-01-02,2016-01-05,2016-01-03,2,7
4,175,4,2016-01-03,2016-01-04,2016-01-05,1,3
5,1324,4,2016-01-03,2016-01-06,2016-01-06,2,6
6,94,4,2016-01-04,2016-01-07,2016-01-05,2,6
7,324,4,2016-01-04,2016-01-07,2016-01-05,2,6
8,1204,4,2016-01-04,2016-01-05,2016-01-05,2,7
9,60,4,2016-01-05,2016-01-08,2016-01-08,1,2
10,442,4,2016-01-05,2016-01-06,2016-01-06,2,6


In [0]:
describe bikestore_project.bronze.orders

col_name,data_type,comment
order_id,bigint,null
customer_id,bigint,null
order_status,bigint,null
order_date,date,null
required_date,date,null
shipped_date,string,null
store_id,bigint,null
staff_id,bigint,null


In [0]:
select * from bikestore_project.bronze.order_items

order_id,item_id,product_id,quantity,list_price,discount
1,1,20,1,599.99,0.2
1,2,8,2,1799.99,0.07
1,3,10,2,1549.0,0.05
1,4,16,2,599.99,0.05
1,5,4,1,2899.99,0.2
2,1,20,1,599.99,0.07
2,2,16,2,599.99,0.05
3,1,3,1,999.99,0.05
3,2,20,1,599.99,0.05
4,1,2,2,749.99,0.1


In [0]:
describe bikestore_project.bronze.order_items

col_name,data_type,comment
order_id,bigint,null
item_id,bigint,null
product_id,bigint,null
quantity,bigint,null
list_price,double,null
discount,double,null


In [0]:
select
o.order_id,
o.item_id,
o.product_id,
o.quantity,
o.list_price as unit_price,
o.discount,
round((o.list_price * o.quantity) * (1-o.discount),2) as total_sale
from bikestore_project.bronze.order_items o

order_id,item_id,product_id,quantity,unit_price,discount,total_sale
1,1,20,1,599.99,0.2,479.99
1,2,8,2,1799.99,0.07,3347.98
1,3,10,2,1549.0,0.05,2943.1
1,4,16,2,599.99,0.05,1139.98
1,5,4,1,2899.99,0.2,2319.99
2,1,20,1,599.99,0.07,557.99
2,2,16,2,599.99,0.05,1139.98
3,1,3,1,999.99,0.05,949.99
3,2,20,1,599.99,0.05,569.99
4,1,2,2,749.99,0.1,1349.98


In [0]:
%python
df_orders = spark.sql("""
with order_totals as (

          select
     o.order_id,
     o.item_id,
     o.product_id,
     o.quantity,
     o.list_price as unit_price,
     o.discount,
     round((o.list_price * o.quantity) * (1-o.discount),2) as total_sale
     from bikestore_project.bronze.order_items o
)




SELECT 
orid.order_id,
ot.total_sale,
csc.customer_id,
csc.city,
CASE WHEN o.order_status = 1 THEN 'Pending'
     WHEN o.order_status = 2 THEN 'Processing'
     WHEN o.order_status = 3 THEN 'Shipped'
     WHEN o.order_status = 4 THEN 'Cancelled'
     ELSE 'Unknown'
END as status,
o.order_status,
o.order_date,
o.required_date,
o.shipped_date,
st.store_id,
stf.staff_id,
stf.first_name
FROM bikestore_project.bronze.orders o
LEFT JOIN bikestore_project.bronze.customers csc on o.customer_id = csc.customer_id
left join bikestore_project.bronze.stores st on o.store_id = st.store_id
left join bikestore_project.bronze.staffs stf on o.staff_id = stf.staff_id
left join bikestore_project.bronze.order_items orid on o.order_id = orid.order_id
left join order_totals ot on o.order_id = ot.order_id
""")

df_orders.write\
    .format('Delta')\
    .mode('overwrite')\
    .saveAsTable('bikestore_project.silver.orders')

In [0]:
%python
display(df_orders)

order_id,total_sale,customer_id,city,status,order_status,order_date,required_date,shipped_date,store_id,staff_id,first_name
1,2319.99,259,Pleasanton,Cancelled,4,2016-01-01,2016-01-03,2016-01-03,1,2,Mireya
2,1139.98,1212,Huntington Station,Cancelled,4,2016-01-01,2016-01-04,2016-01-03,2,6,Marcelene
3,569.99,523,Patchogue,Cancelled,4,2016-01-02,2016-01-05,2016-01-03,2,7,Venita
4,1349.98,175,Duarte,Cancelled,4,2016-01-03,2016-01-04,2016-01-05,1,3,Genna
5,557.99,1324,Utica,Cancelled,4,2016-01-03,2016-01-06,2016-01-06,2,6,Marcelene
6,5579.98,94,Baldwinsville,Cancelled,4,2016-01-04,2016-01-07,2016-01-05,2,6,Marcelene
7,772.2,324,Bellmore,Cancelled,4,2016-01-04,2016-01-07,2016-01-05,2,6,Marcelene
8,1115.98,1204,Saratoga Springs,Cancelled,4,2016-01-04,2016-01-05,2016-01-05,2,7,Venita
9,7199.98,60,San Carlos,Cancelled,4,2016-01-05,2016-01-08,2016-01-08,1,2,Mireya
10,242.99,442,Yonkers,Cancelled,4,2016-01-05,2016-01-06,2016-01-06,2,6,Marcelene
